# Projet Kayak — Étape 3 : scraping des hôtels (Booking.com)

**Objectif** : pour chaque ville, récupérer ses hôtels depuis Booking — nom, note,
prix, URL, et **coordonnées GPS**.

**Pourquoi Playwright et pas `requests` + BeautifulSoup** : Booking charge ses hôtels
via JavaScript et bloque les robots simples (on obtenait un statut `202` et une page
vide). Playwright pilote un vrai navigateur, exécute le JavaScript, et accède au contenu.

**Entrée** : `data/raw/cities.csv` (les 35 villes)
**Sorties** :
- `data/raw/hotels/{city_id}.json` — hôtels bruts, une ville par fichier (cache)
- `data/raw/hotels.csv` — tous les hôtels consolidés



## 1. Imports et configuration

In [ ]:
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
from playwright.async_api import async_playwright

HOTELS_DIR = Path("data/raw/hotels")
HOTELS_DIR.mkdir(parents=True, exist_ok=True)

# Dates de séjour pour la recherche
CHECKIN = "2026-08-01"
CHECKOUT = "2026-08-02"

USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/120.0.0.0 Safari/537.36")

## 2. Fonctions de nettoyage

Le scraping renvoie du texte tel qu'affiché pour un humain (`"€ 297"`, `"Avec une
note de 8,3 | ..."`). On le transforme en données exploitables. Chaque fonction a été
validée sur les vraies données de Booking.

In [ ]:
def extraire_note(texte):
    """'Avec une note de 8,3 | ...' -> 8.3 ; gère aussi '10' (entier)."""
    if not texte:
        return None
    m = re.search(r"(\d{1,2}(?:,\d)?)", texte)
    return float(m.group(1).replace(",", ".")) if m else None


def extraire_prix(texte):
    """'€ 297' ou '€ 1 297' -> 297.0 / 1297.0"""
    if not texte:
        return None
    chiffres = re.sub(r"[^\d]", "", texte)
    return float(chiffres) if chiffres else None


def nettoyer_url(url):
    """Retire le tracking '?aid=...'."""
    return url.split("?")[0] if url else None


def extraire_latlng(html):
    """Lit data-atlas-latlng=\"lat,lon\" sur la page hôtel -> (lat, lon)."""
    m = re.search(r'data-atlas-latlng="(-?\d+\.\d+),(-?\d+\.\d+)"', html)
    return (float(m.group(1)), float(m.group(2))) if m else (None, None)

## 3. Le scraper d'une ville

Deux temps :
1. **Page de résultats** → nom, note, prix, URL des hôtels (1 page, rapide).
2. **Page de chaque hôtel** → coordonnées GPS via `data-atlas-latlng`.

On limite à `max_hotels` par ville pour maîtriser le temps (25 suffit largement pour
le livrable « top 20 hôtels »).

In [ ]:
async def scraper_ville(ville, max_hotels=25):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=USER_AGENT,
            viewport={"width": 1280, "height": 900},
            locale="fr-FR",
        )
        page = await context.new_page()

        # --- ÉTAPE 1 : page de résultats ---
        url = ("https://www.booking.com/searchresults.fr.html"
               f"?ss={ville}&checkin={CHECKIN}&checkout={CHECKOUT}"
               "&group_adults=2&no_rooms=1&group_children=0")
        await page.goto(url, timeout=60000, wait_until="domcontentloaded")
        try:
            await page.wait_for_selector('[data-testid="property-card"]', timeout=25000)
        except Exception:
            print(f"Aucune carte pour {ville}")
            await browser.close()
            return []

        cartes = await page.query_selector_all('[data-testid="property-card"]')
        cartes = cartes[:max_hotels]

        hotels = []
        for carte in cartes:
            nom_el = await carte.query_selector('[data-testid="title"]')
            lien_el = await carte.query_selector('a[data-testid="title-link"]')
            note_el = await carte.query_selector('[data-testid="review-score"]')
            prix_el = await carte.query_selector('[data-testid="price-and-discounted-price"]')
            hotels.append({
                "ville": ville,
                "nom": await nom_el.inner_text() if nom_el else None,
                "url": nettoyer_url(await lien_el.get_attribute("href")) if lien_el else None,
                "note": extraire_note(await note_el.inner_text() if note_el else None),
                "prix": extraire_prix(await prix_el.inner_text() if prix_el else None),
            })

        # --- ÉTAPE 2 : coordonnées via la page de chaque hôtel ---
        page_hotel = await context.new_page()
        for hotel in hotels:
            if not hotel["url"]:
                hotel["lat"], hotel["lon"] = None, None
                continue
            try:
                await page_hotel.goto(hotel["url"], timeout=45000, wait_until="domcontentloaded")
                await page_hotel.wait_for_timeout(1500)
                html = await page_hotel.content()
                hotel["lat"], hotel["lon"] = extraire_latlng(html)
            except Exception:
                hotel["lat"], hotel["lon"] = None, None
            await asyncio.sleep(1)  

        await browser.close()
        return hotels

## 4. La boucle sur les 35 villes, avec cache



In [ ]:
df_cities = pd.read_csv("data/raw/cities.csv")

for row in df_cities.itertuples():
    cache_file = HOTELS_DIR / f"{row.city_id}.json"

    if cache_file.exists():                       
        print(f"[CACHE] {row.city_id:>2}. {row.city}")
        continue

    print(f"[SCRAPE] {row.city_id:>2}. {row.city} ...")
    hotels = await scraper_ville(row.city)

   
    for h in hotels:
        h["city_id"] = row.city_id

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(hotels, f, ensure_ascii=False, indent=2)

    n_coords = sum(1 for h in hotels if h.get("lat"))
    print(f"         -> {len(hotels)} hôtels, {n_coords} avec coordonnées")
    await asyncio.sleep(2)  

print("\nScraping terminé.")

[SCRAPE]  1. Mont Saint Michel ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  2. St Malo ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  3. Bayeux ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  4. Le Havre ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  5. Rouen ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  6. Paris ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  7. Amiens ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  8. Lille ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE]  9. Strasbourg ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 11. Colmar ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 12. Eguisheim ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 13. Besancon ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 14. Dijon ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 15. Annecy ...
         -> 25 hôtels, 25 avec coordonnées
[SCRAPE] 16. Grenoble ...
         -> 25 hô

## 5. Consolidation en un seul DataFrame

On relit tous les fichiers JSON du cache et on les empile.

In [ ]:
tous_hotels = []
for cache_file in sorted(HOTELS_DIR.glob("*.json")):
    with open(cache_file, encoding="utf-8") as f:
        tous_hotels.extend(json.load(f))

df_hotels = pd.DataFrame(tous_hotels)
print(f"Total : {len(df_hotels)} hôtels sur {df_hotels['city_id'].nunique()} villes")
print(f"Avec coordonnées : {df_hotels['lat'].notna().sum()} / {len(df_hotels)}")
df_hotels.head(10)

Total : 875 hôtels sur 35 villes
Avec coordonnées : 874 / 875


,ville,nom,url,note,prix,lat,lon,city_id
0,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html,8.1,None,48.614700,-1.509617,1
1,Mont Saint Michel,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,8.1,None,48.617587,-1.510396,1
2,Mont Saint Michel,Le Saint Aubert,https://www.booking.com/hotel/fr/hotel-saint-a...,7.4,None,48.612938,-1.510105,1
3,Mont Saint Michel,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,8.4,None,48.614247,-1.510545,1
4,Mont Saint Michel,La Confiance,https://www.booking.com/hotel/fr/les-terrasses...,7.5,None,48.635300,-1.510397,1
5,Mont Saint Michel,Hotel De La Digue,https://www.booking.com/hotel/fr/de-la-digue.f...,7.5,None,48.616882,-1.510918,1
6,Mont Saint Michel,Hotel Gabriel,https://www.booking.com/hotel/fr/hotel-gabriel...,8.1,None,48.615381,-1.510710,1
7,Mont Saint Michel,La Mère Poulard,https://www.booking.com/hotel/fr/la-mere-poula...,7.6,None,48.635058,-1.510944,1
8,Mont Saint Michel,La Vieille Auberge,https://www.booking.com/hotel/fr/la-vieille-au...,7.5,None,48.636063,-1.511457,1
9,Mont Saint Michel,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,8.3,None,48.616263,-1.510906,1


## 6. Contrôle qualité

In [ ]:
# Combien d'hôtels par ville ?
print("Hôtels par ville :")
print(df_hotels.groupby("city_id").size().describe())

# Lignes sans nom ou sans coordonnées
print("\nSans nom :", df_hotels["nom"].isna().sum())
print("Sans coordonnées :", df_hotels["lat"].isna().sum())
print("Sans note :", df_hotels["note"].isna().sum())

Hôtels par ville :
count    35.0
mean     25.0
std       0.0
min      25.0
25%      25.0
50%      25.0
75%      25.0
max      25.0
dtype: float64

Sans nom : 0
Sans coordonnées : 1
Sans note : 25


In [ ]:
## 7. Sauvegarde

output = Path("data/raw/hotels.csv")
df_hotels.to_csv(output, index=False)
print("Écrit :", output, "|", len(df_hotels), "hôtels")

Écrit : data/raw/hotels.csv | 875 hôtels


In [ ]:
import pandas as pd

df_hotels = pd.read_csv("data/raw/hotels.csv")
print("Avant :", len(df_hotels))

# On garde la première occurrence de chaque URL
df_hotels = df_hotels.drop_duplicates(subset="url", keep="first").reset_index(drop=True)
print("Après :", len(df_hotels))

df_hotels.to_csv("data/raw/hotels.csv", index=False)

Avant : 875
Après : 875
